In [108]:
import pandas as pd
import unicodedata
import time
import nltk
import re
from nltk.stem import WordNetLemmatizer
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import ComplementNB

SEED = 42


In [59]:
# 读取训练集和测试集
train_df = pd.read_json('data/train.json')
test_df = pd.read_json('data/test.json')

In [60]:
try:
    nltk.data.find('corpora/wordnet.zip')
except LookupError:
    print("正在下载 NLTK 数据...")
    nltk.download('wordnet')
    nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

In [105]:
STOP_WORDS = {
    # 3. 饮食与健康描述 (重灾区：low-fat, gluten-free, etc.)
    'low', 'fat', 'non', 'skim', 'part', 'reduced', 'less', 'sodium', 'free', 'gluten',
    'diet', 'lite', 'light', 'lean', 'extra', 'no', 'added', 'calorie',
    'sugar', 'sweetened', 'unsweetened',

    # 4. 食材状态/预处理 (形容词)
    'chopped', 'diced', 'sliced', 'fresh', 'minced', 'large', 'small', 'medium', 'jumbo',
    'package', 'can', 'canned', 'container', 'box', 'bag', 'packet',
    'frozen', 'melted', 'beaten', 'ground', 'crushed', 'whole',
    'boneless', 'skinless', 'bone', 'skin', 'chunk', 'chunks',
    'shredded', 'grated', 'crumbled', 'flake', 'flaked', # [新增] 来自 shredded cheese
    'soft', 'softened', 'hard', 'boiled', # [新增] hard-boiled egg -> egg
    'active', 'dry', 'dried', # [新增] active yeast, dried basil -> yeast, basil

    # 5. 商业/营销/通用术语
    'brand', 'store', 'bought', 'ready', 'made', 'instant',
    'original', 'style', 'condensed', 'evaporated', # [新增] condensed milk -> milk
    'mix', 'mixture', 'blend', # [新增] seasoning mix -> seasoning
    'flavor', 'flavored', # [新增] chicken flavored -> chicken
    'old', 'fashioned', # [新增] old-fashioned oats -> oats
    'purpose', 'rising', # [新增] all-purpose, self-rising

    # 6. 逗号后面的动作 (动词)
}

MY_STOP_WORDS = {
    'fresh', 'ground', 'spread',

    # 'min', 'any', 'v8', 'vol', 'spread', 'reserved', 'half', 'inch',

    'any',

    'drain', 'drained',
    'undrain', 'undrained', 'dry', 'dried',
    'cook', 'cooked', 'cooking', 'uncooked',
    'rins',
    'peel', 'peeled',
    'devein', 'deveined',
    'thaw', 'thawed',
    'cut',
    'prepare', 'prepared',
    'trimmed',
    'scrub', 'scrubbed',
    'well',
    'squeeze', 'squeezed'

}

def clean_pipeline(ingredient_list):
    clean_ingredients = []

    for item in ingredient_list:
        # 字符标准化
        item = unicodedata.normalize('NFKD', item).encode('ascii', 'ignore').decode('utf-8')

        # 转小写
        item = item.lower()

        # 处理 half & half 为一个词
        if 'half & half' in item:
            item = item.replace('half & half', 'halfandhalf')

        item = item.replace('-', ' ').replace('&', ' ')

        item = re.sub(r'[^a-z]', ' ', item)

        # 词形还原
        words = item.split()
        words = [word for word in words if word not in MY_STOP_WORDS]
        words = [lemmatizer.lemmatize(word, pos='n') for word in words]

        cleaned_item = " ".join(words)

        if cleaned_item:
            clean_ingredients.append(cleaned_item)

    return ' '.join(clean_ingredients)


# --- 执行清洗 ---
print("开始清洗训练集...")
train_df['ingredients_clean'] = train_df['ingredients'].apply(clean_pipeline)
print("开始清洗测试集...")
test_df['ingredients_clean'] = test_df['ingredients'].apply(clean_pipeline)


开始清洗训练集...
开始清洗测试集...


In [102]:

vectorizer = TfidfVectorizer(
    stop_words='english',
    ngram_range=(1, 2),
    binary=True,          # 只要出现过就算，不管几次
    min_df=3,             # 出现少于 3 次的词直接扔掉
    sublinear_tf=True
)

X = vectorizer.fit_transform(train_df['ingredients_clean'])
X_submission = vectorizer.transform(test_df['ingredients_clean'])

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(train_df['cuisine'])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=SEED)



feature_names = vectorizer.get_feature_names_out()

# 2. 统计频次
# X 是稀疏矩阵，getnnz(axis=0) 方法可以极其快速地计算每一列有多少个非零元素
# 这代表了“该食材在多少个食谱中出现过” (Document Frequency)
# 即使你设置了 binary=True，getnnz 依然统计的是出现的次数
occurrences = X.getnnz(axis=0)

# 3. 创建 DataFrame 进行展示
df_freq = pd.DataFrame({
    'term': feature_names,
    'count': occurrences
})

# 4. 按出现次数降序排列
df_freq = df_freq.sort_values(by='count', ascending=False)

print("-" * 30)
print(f"特征总数量: {len(feature_names)}")
print("-" * 30)

print("【出现频率最高的 20 个食材/词组】")
print(df_freq.head(20).to_string(index=False)) # to_string 去掉索引，打印更干净

print("\n" + "-" * 30 + "\n")

print("【出现频率最低的 20 个食材/词组】")
# 这些通常是刚好达到你 min_df=3 门槛的词
print(df_freq.tail(20).to_string(index=False))
print("-" * 30)
# ================= 插入结束 =================


print(f"标签向量形状 (y): {y.shape}")

# 1. 选择要调优的模型：
# LogisticRegression 最佳 C 值: 4.0
# MODEL_TO_TUNE = LogisticRegression(
#     solver='lbfgs',      # 默认求解器
#     max_iter=1000,       # 迭代次数
#     random_state=SEED,
#     n_jobs=-1            # 利用所有 CPU 核心加速
# )
# PARAM_GRID = {'C': [3.5, 4.0, 4.5, 5.0, 5.5, 6.0]}

# LinearSVC 最佳 C 值: 0.5
# MODEL_TO_TUNE = LinearSVC(
#     dual='auto',
#     max_iter=1000,
#     random_state=42,
# )
# PARAM_GRID = {'C': [3.5, 4.0, 4.5, 5.0, 5.5, 6.0]}

# 多项式朴素贝叶斯 最佳 alpha 值: 0.05
# MODEL_TO_TUNE = MultinomialNB()
# PARAM_GRID = {'alpha': [0.045, 0.05, 0.06]}

# 补充朴素贝叶斯 最佳 alpha 值: 0.35
MODEL_TO_TUNE = ComplementNB()
PARAM_GRID = {'alpha': [0.3, 0.35, 0.4]}

# 设置交叉验证折数 (cv)：3 或 5 是常见选择
CV_FOLDS = 3


# 通用调优逻辑
if 'X_train' not in locals() or 'y_train' not in locals():
    print("错误：请确保 X_train 和 y_train 变量已在前面的单元格中定义。")
else:
    model_name = type(MODEL_TO_TUNE).__name__
    print(f"--- 🚀 正在对模型 {model_name} 进行参数调优 (CV={CV_FOLDS}) ---")

    start_time = time.time()

    # 实例化 GridSearchCV
    grid_search = GridSearchCV(
        estimator=MODEL_TO_TUNE,
        param_grid=PARAM_GRID,
        cv=CV_FOLDS,
        n_jobs=-1,  # 使用所有 CPU 核心加速计算
        scoring='accuracy',
        verbose=1   # 打印进度
    )


    # 开始搜索
    grid_search.fit(X_train, y_train)

    end_time = time.time()

    # 提取最佳结果
    best_para = grid_search.best_params_['alpha']
    best_score_cv = grid_search.best_score_

    # 使用最佳参数的模型对验证集进行最终评估
    best_model = grid_search.best_estimator_
    y_pred_val = best_model.predict(X_val)
    final_val_accuracy = accuracy_score(y_val, y_pred_val)

    # --- 打印报告 ---
    print("\n==============================================")
    print(f"✅ {model_name} 调优报告:")
    print("==============================================")
    print(f"最佳参数值: {best_para}")
    print(f"交叉验证 (CV={CV_FOLDS}) 最佳平均准确率: {best_score_cv:.4f}")
    print(f"验证集 (Validation Set) 最终准确率: {final_val_accuracy:.4f}")
    print(f"总耗时: {end_time - start_time:.2f} 秒")
    print("==============================================")

In [109]:

# 定义并训练逻辑回归模型 (Logistic Regression)
# C : 正则化强度。C越小，正则化越强（防止过拟合）；C越大，越拟合训练数据。
# model = LogisticRegression(
#     solver='lbfgs',      # 默认求解器
#     max_iter=1000,       # 迭代次数
#     C=4,
#     random_state=SEED,
#     n_jobs=-1            # 利用所有 CPU 核心加速
# )

model_bert = LinearSVC(
    dual='auto',
    max_iter=1000,
    C=0.5,
    random_state=SEED,
)

model_bert.fit(X_train, y_train)

y_pred = model_bert.predict(X_val)

# 计算准确率
accuracy = accuracy_score(y_val, y_pred)
print("-" * 30)
print(f"验证集准确率 (Accuracy): {accuracy:.4f}") # 保留4位小数
print("-" * 30)

# 打印详细的分类报告 查看每个菜系的 F1-score
# target_names 让报告显示真实的菜系名
from sklearn.metrics import classification_report
print("详细分类报告:")
print(classification_report(y_val, y_pred, target_names=label_encoder.classes_))

------------------------------
验证集准确率 (Accuracy): 0.7921
------------------------------
详细分类报告:
              precision    recall  f1-score   support

   brazilian       0.79      0.59      0.68        71
     british       0.60      0.41      0.48       116
cajun_creole       0.78      0.70      0.74       227
     chinese       0.78      0.88      0.83       398
    filipino       0.76      0.63      0.69       134
      french       0.61      0.64      0.63       410
       greek       0.83      0.72      0.77       168
      indian       0.84      0.91      0.87       502
       irish       0.59      0.46      0.52       106
     italian       0.80      0.91      0.85      1293
    jamaican       0.87      0.65      0.74        96
    japanese       0.86      0.69      0.77       239
      korean       0.86      0.71      0.78       150
     mexican       0.90      0.93      0.92       964
    moroccan       0.81      0.72      0.76       121
     russian       0.50      0.29      

In [104]:
model_bert.fit(X, y)
X_test_final = vectorizer.transform(test_df['ingredients_clean'])

y_pred_indices = model_bert.predict(X_test_final)

y_pred_text = label_encoder.inverse_transform(y_pred_indices)

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'cuisine': y_pred_text
})

filename = 'submission.csv'
submission_df.to_csv(filename, index=False)

print(f"文件已生成: {filename}")

文件已生成: submission6.csv


In [47]:

import xgboost as xgb
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder

# =========================================================
# 💡 【用户配置区】 设置降维维度
# =========================================================
# 500 维通常是一个很好的起点，既保留了大部分信息，又减少了计算量。
N_COMPONENTS = 300

# =========================================================
# 1. 降维操作 (Truncated SVD)
# =========================================================
print(f"1. 正在使用 Truncated SVD 将特征维度降至 {N_COMPONENTS}...")

# 初始化 SVD 模型
svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=SEED)

# 对训练集进行拟合和转换
X_train_svd = svd.fit_transform(X_train)
# 对验证集进行转换
X_val_svd = svd.transform(X_val)

print(f"原始特征维度: {X_train.shape[1]} -> 降维后特征维度: {X_train_svd.shape[1]}")

# =========================================================
# 2. 准备标签 (XGBoost 标签必须从 0 开始编码)
# =========================================================
# 假设你之前用 LabelEncoder 将字符串标签转成了数字。
# 确保你使用的是 LabelEncoder 的实例 (这里我们假设它是可用的)
# XGBoost 默认 objective='multi:softmax' 要求标签从 0 到 num_class-1
if 'label_encoder' in locals():
    # 确保 y_train 和 y_val 是数值编码
    y_train_encoded = y_train
    y_val_encoded = y_val
    num_classes = len(label_encoder.classes_)
else:
    # 如果没有 LabelEncoder，需要重新编码 (假设 y_train 和 y_val 是字符串)
    print("警告: 正在重新编码 y 标签。")
    le = LabelEncoder()
    y_train_encoded = le.fit_transform(y_train)
    y_val_encoded = le.transform(y_val)
    num_classes = len(le.classes_)


# =========================================================
# 3. XGBoost 模型训练
# =========================================================
print("\n2. 正在训练 XGBoost 模型...")
start_time = time.time()

xgb_model = xgb.XGBClassifier(
    objective='multi:softmax',
    num_class=num_classes,
    n_estimators=1000,       # 树的数量，一个好的起点
    learning_rate=0.15,      # 学习率，可以调优
    max_depth=5,            # 树的最大深度，可以调优
    eval_metric='merror',   # 评估指标 (多分类错误率)
    random_state=SEED,
    n_jobs=-1               # 利用所有核心
)

# 使用降维后的数据进行拟合
xgb_model.fit(X_train_svd, y_train_encoded)

end_time = time.time()

# =========================================================
# 4. 预测与评估
# =========================================================
print("\n3. 正在预测验证集...")
y_pred_xgb_encoded = xgb_model.predict(X_val_svd)
acc_xgb = accuracy_score(y_val_encoded, y_pred_xgb_encoded)

print("-" * 30)
print(f"XGBoost (SVD={N_COMPONENTS}) 验证集准确率: {acc_xgb:.4f}")
print(f"训练总耗时: {end_time - start_time:.2f} 秒")
print("-" * 30)

# 打印混淆报告 (可选，需要将预测结果解码回字符串标签)
# y_pred_xgb_decoded = label_encoder.inverse_transform(y_pred_xgb_encoded)
# print(classification_report(y_val, y_pred_xgb_decoded))

1. 正在使用 Truncated SVD 将特征维度降至 1000...
原始特征维度: 14887 -> 降维后特征维度: 1000

2. 正在训练 XGBoost 模型...

3. 正在预测验证集...
------------------------------
XGBoost (SVD=1000) 验证集准确率: 0.7593
训练总耗时: 1875.75 秒
------------------------------
